# 10 — First-k / Lead (baseline posizionale)

**First-k** (o *Lead*) è la baseline più semplice della summarization estrattiva: per ogni
articolo del cluster si prendono le prime `k` frasi e le si concatena. Si fonda sul *lead bias*
della scrittura giornalistica (piramide rovesciata): le informazioni più salienti tendono a
comparire in apertura. Con `k = 3` corrisponde alla baseline **First-3** del paper Multi-News
(Fabbri et al., 2019), che sul dataset supera empiricamente LexRank, TextRank e MMR. È il termine
di paragone che ogni metodo più complesso dovrebbe battere per giustificarsi.

Nel quadro della lezione del Master la copertura è *parziale*: la posizione della frase compare
come feature classica di scoring (*sentence position*), non come baseline autonoma.

## Due varianti di segmentazione (confronto)

First-k richiede solo di **dividere ogni articolo in frasi**: qui questo passaggio è l'oggetto
dell'esperimento. Lo eseguiamo con due segmentatori diversi e ne confrontiamo le metriche:

- **`firstk_psr`** — usa la segmentazione di `psr.summarization` (per punteggiatura), la
  **stessa** di TextRank/LexRank (notebook 01/02): confronto equo con gli altri estrattivi.
- **`firstk_nltk`** — usa `nltk.sent_tokenize` (modello *Punkt*, gestisce le abbreviazioni tipo
  "U.S.", "Dr."): segmentazione più raffinata, per capire se migliora le metriche MDS.

La logica First-k è identica nelle due varianti (prime `K_SENTENCES` frasi per articolo,
concatenate); cambia **solo** il modo di individuare i confini di frase. La **valutazione** è
quella condivisa da tutti i metodi del benchmark (ROUGE/BLEU/METEOR di pyAutoSummarizer), quindi i
punteggi restano confrontabili. I due segmentatori sono alternative: ogni variante ne usa uno solo.

Come i notebook 01/02, opera su due ambiti (`SCOPE`): `sample` (campione condiviso) e `full`
(intero `complete.tab` in streaming). Essendo una baseline senza modelli né ranking, è
velocissima anche su CPU. Riassunti salvati incrementalmente con **ripresa**; metriche
ricalcolabili dai soli file salvati.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

# Dati del tokenizzatore di frasi di nltk (Punkt), usati dalla variante firstk_nltk.
# 'punkt_tab' è richiesto dalle versioni recenti di nltk; scarichiamo entrambi, in silenzio.
import nltk
for _risorsa in ('punkt', 'punkt_tab'):
    try:
        nltk.download(_risorsa, quiet=True)
    except Exception:
        pass

In [ ]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

SCOPE       = 'sample'   # 'sample' = campione condiviso | 'full' = intero complete.tab
N_SAMPLES   = 100        # deve combaciare con il campione creato dal notebook 00
SEED        = 42
LIMIT       = None       # es. 3 per uno smoke test rapido; None = tutti
K_SENTENCES = 3          # prime k frasi per articolo (k=3 = baseline First-3 del paper)
STOP_WORDS  = ['en']     # solo per istanziare psr.summarization; l'output resta il testo originale

# Le due varianti: stesso First-k, segmentatore di frasi diverso
METODI = ['firstk_psr', 'firstk_nltk']

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'

print(f'Ambito (SCOPE)  : {SCOPE}')
print(f'Frasi/articolo  : {K_SENTENCES}')
print(f'Varianti        : {METODI}')

## Generazione dei riassunti

Per ogni documento si separano gli articoli sul token `` ||||| `` (per questo si passa
`prepara=str.strip` al ciclo condiviso, così `genera` vede il separatore grezzo, invece del
default che lo sostituisce con un newline). Da ciascun articolo si estraggono le prime
`K_SENTENCES` frasi — con `psr.summarization` per la variante `firstk_psr`, con
`nltk.sent_tokenize` per `firstk_nltk` — e si concatenano. Nessun modello, nessun ranking.

Le due varianti vengono generate in sequenza in file separati
(`results/summaries/{metodo}_{scope}.tsv`), ciascuna con **ripresa** automatica.

In [ ]:
from pyAutoSummarizer.base import psr
from nltk.tokenize import sent_tokenize


def estrai_psr(articolo):
    """Prime K_SENTENCES frasi dell'articolo con la segmentazione di psr.summarization.

    `s.original` e' la lista delle frasi originali in ordine di documento (stesso split
    abbreviation-aware di TextRank/LexRank). Le si legge direttamente per mantenere l'ordine:
    `show_summary` invece ordina per rank decrescente e restituirebbe le frasi invertite."""
    s = psr.summarization(articolo, stop_words=STOP_WORDS)
    frasi = [f.strip() for f in getattr(s, 'original', []) if f and f.strip()]
    return ' '.join(frasi[:K_SENTENCES])


def estrai_nltk(articolo):
    """Prime K_SENTENCES frasi dell'articolo con nltk.sent_tokenize (Punkt)."""
    frasi = sent_tokenize(articolo)[:K_SENTENCES]
    return ' '.join(f.strip() for f in frasi if f.strip())


def make_genera(estrai_frasi):
    """Crea la funzione genera(documento) per una data strategia di estrazione per articolo."""
    def genera(documento):
        parti = []
        for articolo in documento.split(su.SEPARATORE_ARTICOLI):
            articolo = articolo.strip()
            if not articolo:
                continue
            testo = estrai_frasi(articolo)
            if testo:
                parti.append(testo)
        return ' '.join(parti)
    return genera


ESTRATTORI = {'firstk_psr': estrai_psr, 'firstk_nltk': estrai_nltk}

for metodo in METODI:
    out_path = P['summaries_dir'] / f'{metodo}_{SCOPE}.tsv'
    esempi = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
              else su.itera_complete_tab(P['complete_tab']))
    scrittore = su.ScrittoreRiassunti(out_path)
    print(f'== {metodo} -> {out_path.name} ==')
    su.ciclo_summarization(esempi, scrittore, make_genera(ESTRATTORI[metodo]),
                           limit=LIMIT, etichetta=f'{metodo} ', prepara=str.strip)
    scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con la normalizzazione identica a tutti i metodi del
benchmark. Gli `overall` delle due varianti vengono stampati affiancati per il confronto diretto
**psr vs nltk**. Output: `results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`.

In [ ]:
import json

overall = {}
for metodo in METODI:
    out_path = P['summaries_dir'] / f'{metodo}_{SCOPE}.tsv'
    riassunti   = su.carica_riassunti(out_path)
    riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
                   else su.itera_complete_tab(P['complete_tab']))
    segmentatore = 'nltk (Punkt)' if metodo == 'firstk_nltk' else 'psr.summarization'
    config = {'k_sentences_per_articolo': K_SENTENCES, 'variante': 'per-articolo',
              'segmentatore': segmentatore, 'libreria_valutazione': 'pyAutoSummarizer 1.2.0'}
    righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, metodo, SCOPE,
                                         P['metrics_dir'], config)
    overall[metodo] = aggregato['overall']

print('\nConfronto psr vs nltk (medie complessive):')
print(json.dumps(overall, indent=2))

## Ispezione qualitativa

Per ogni esempio del campione vengono mostrati **per intero** (nessun troncamento) il documento
sorgente (gli articoli separati da `` ||||| ``), il riassunto di riferimento e il riassunto
generato da **entrambe** le varianti, così da confrontare a colpo d'occhio dove i due segmentatori
tagliano le frasi in modo diverso. I due parametri in testa alla cella regolano l'output:
`N_ISPEZIONE` (`None` = tutti gli esempi) e `MOSTRA_DOCUMENTO` (articoli sorgente sì/no).

In [ ]:
# --- Ispezione qualitativa completa -----------------------------------------
# Mostra per intero (nessun troncamento): documento sorgente, riferimento e
# riassunto di CIASCUNA variante, per confrontare psr vs nltk a colpo d'occhio.
N_ISPEZIONE      = None   # quanti esempi: un intero (es. 5) oppure None = tutti
MOSTRA_DOCUMENTO = True   # False per nascondere gli articoli sorgente (output piu' corto)

riferimenti = (su.carica_campione(SAMPLE_PATH) if SCOPE == 'sample'
               else su.itera_complete_tab(P['complete_tab']))
riassunti = {m: su.carica_riassunti(P['summaries_dir'] / f'{m}_{SCOPE}.tsv') for m in METODI}

mostrati = 0
for rif in riferimenti:
    rid = rif['row_id']
    if not all(rid in riassunti[m] for m in METODI):
        continue
    if N_ISPEZIONE is not None and mostrati >= N_ISPEZIONE:
        break
    print('=' * 100)
    print(f"row_id={rid} | split={rif['split']}")
    if MOSTRA_DOCUMENTO:
        articoli = [a.strip() for a in rif['document'].split(su.SEPARATORE_ARTICOLI) if a.strip()]
        print(f"\n----- DOCUMENTO ({len(articoli)} articoli) -----")
        for i, art in enumerate(articoli, 1):
            print(f"[articolo {i}]\n{art}\n")
    print(f"----- RIFERIMENTO -----\n{su.pulisci_riferimento(rif['summary'])}\n")
    for m in METODI:
        print(f"----- GENERATO [{m}] -----\n{riassunti[m][rid]}\n")
    mostrati += 1
print(f"\n({mostrati} esempi mostrati)")